# TabFM v1.0.0 (Google Tabular Foundation Model)

Source: https://github.com/google-research/tabfmUses

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

df = pd.read_csv("total_gva_engineered_features.csv")

target = "log_total_GVA_2023"
feats = [c for c in df.columns if c not in ["LSOA21CD", "MSOA21CD", "is_swindon", target]]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, df[target], groups=df["MSOA21CD"]))
train, test = df.iloc[train_idx].reset_index(drop=True), df.iloc[test_idx].reset_index(drop=True)

X_tr, X_te = train[feats].values, test[feats].values
y_tr, y_te = train[target].values, test[target].values

print("train", train.shape)
print("test", test.shape)
print("features", feats)

train (903, 14)
test (222, 14)
features ['log_voa_rv_2023', 'rv_per_working_age', 'sme_density', 'qualification_index', 'firm_size_diversity', 'rv_per_employee', 'sme_qual_interaction', 'employment_quality', 'modern_sector_leverage', 'asset_growth_diversity']


In [3]:
from huggingface_hub import snapshot_download
from safetensors.torch import load_file
from tabfm import TabFMRegressor
from tabfm.src.pytorch.model import TabFM
from tabfm.src.pytorch.tabfm_v1_0_0 import RegressionConfig

base_path = snapshot_download(repo_id="google/tabfm-1.0.0-pytorch")
state_dict = load_file(f"{base_path}/regression/model.safetensors")

config = RegressionConfig()
model = TabFM(**config.to_dict())
model.load_state_dict(state_dict, strict=True)
model.eval()

reg = TabFMRegressor(model=model)

Fetching 8 files: 100%|██████████| 8/8 [00:00<?, ?it/s]


In [4]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score

reg.fit(X_tr, y_tr)
pred = reg.predict(X_te)

print(f"target: {target}")
print(f"MAE = {mean_absolute_error(y_te, pred):.4f}")
print(f"RMSE = {np.sqrt(mean_squared_error(y_te, pred)):.4f}")
print(f"MAPE = {mean_absolute_percentage_error(y_te, pred):.4f}")
print(f"R2 = {r2_score(y_te, pred):.4f}")

target: log_total_GVA_2023
MAE = 0.4156
RMSE = 0.6718
MAPE = 0.1167
R2 = 0.6271


## Swindon-only robustness check

In [5]:
for group_name, mask in [("Swindon", test["is_swindon"] == "Swindon"), ("Others", test["is_swindon"] == "Others")]:
    if mask.sum() == 0:
        continue
    y_sub, pred_sub = y_te[mask.values], pred[mask.values]
    print(f"[{group_name}] n={mask.sum()} MAE={mean_absolute_error(y_sub, pred_sub):.4f} R2={r2_score(y_sub, pred_sub):.4f}")

[Swindon] n=18 MAE=0.2344 R2=0.7913
[Others] n=204 MAE=0.4316 R2=0.6103
